In [1]:
import os
import nibabel as nib

input_folder = '/home/jaimebarranco/Desktop/tmp/test_inference_flipped_images/flipped_images_nnUNet/output'
output_folder = '/home/jaimebarranco/Desktop/tmp/test_inference_flipped_images/flipped_images_nnUNet/output_flipped_back'

# Ensure the output folder exists
os.makedirs(output_folder, exist_ok=True)

# Function to flip the axis of the image back to the original orientation
def flip_axis_back(img):
    data = img.get_fdata()
    flipped_back_data = data[::-1, :, :]
    flipped_back_img = nib.Nifti1Image(flipped_back_data, img.affine, img.header)
    return flipped_back_img

# Iterate through the input folder to find NIfTI files
for root, dirs, files in os.walk(input_folder):
    for file in files:
        if file.endswith('.nii.gz'):
            nifti_path = os.path.join(root, file)
            
            # Load the NIfTI image
            img = nib.load(nifti_path)
            
            # Flip the axis back
            flipped_back_img = flip_axis_back(img)
            
            # Create the same folder structure in the output folder
            relative_path = os.path.relpath(root, input_folder)
            subject_folder = os.path.join(output_folder, relative_path)
            os.makedirs(subject_folder, exist_ok=True)
            
            # Save the flipped back image
            flipped_back_nifti_path = os.path.join(subject_folder, os.path.basename(nifti_path).replace('.nii.gz', '_flipped_back.nii.gz'))
            nib.save(flipped_back_img, flipped_back_nifti_path)

In [1]:
import os
import nibabel as nib
import numpy as np
import random
import shutil

images_input = "/mnt/sda1/Repos/a-eye/Data/SHIP_dataset/non_labeled_dataset/non_labeled_dataset_nifti_nnunet"
left_folder_input = "/mnt/sda1/Repos/a-eye/a-eye_segmentation/deep_learning/nnUNet/nnUNet/nnUNet_inference/nnUNet_inference_non_labeled_dataset_left_eyes"
right_folder_input = "/mnt/sda1/Repos/a-eye/a-eye_segmentation/deep_learning/nnUNet/nnUNet/nnUNet_inference/nnUNet_inference_non_labeled_dataset"
merged_folder_input = "/mnt/sda1/Repos/a-eye/a-eye_segmentation/deep_learning/nnUNet/nnUNet/nnUNet_inference/nnUNet_inference_non_labeled_dataset_merged_segs"

images_output = "/mnt/sda1/Repos/a-eye/Data/SHIP_dataset/non_labeled_dataset/evaluation_left_eyes/images"
left_folder_output = "/mnt/sda1/Repos/a-eye/Data/SHIP_dataset/non_labeled_dataset/evaluation_left_eyes/left_seg"
right_folder_output = "/mnt/sda1/Repos/a-eye/Data/SHIP_dataset/non_labeled_dataset/evaluation_left_eyes/right_seg"
merged_folder_output = "/mnt/sda1/Repos/a-eye/Data/SHIP_dataset/non_labeled_dataset/evaluation_left_eyes/merged_seg"

# Create output directories
os.makedirs(images_output, exist_ok=True)
os.makedirs(left_folder_output, exist_ok=True)
os.makedirs(right_folder_output, exist_ok=True)
os.makedirs(merged_folder_output, exist_ok=True)

# Get all files from images_input
all_files = [f for f in os.listdir(images_input) if f.endswith('.nii.gz')]

# Select 10 random cases
random.seed(42)  # For reproducibility
selected_files = random.sample(all_files, min(10, len(all_files)))


In [2]:
import os

temp_folder = '/mnt/sda1/Repos/a-eye/Data/SHIP_dataset/new_manual_annotations/images/temp_inference_cropped_images_left_quadrant/left_segs'

for file in os.listdir(temp_folder):
    if file.endswith('.nii.gz'):
        # Extract the first chunk before the first underscore
        new_name = file.split('_')[0] + '.nii.gz'
        old_path = os.path.join(temp_folder, file)
        new_path = os.path.join(temp_folder, new_name)
        os.rename(old_path, new_path)

In [2]:
import pandas as pd
import numpy as np

# Paths
csv_metadata = '/mnt/sda1/Repos/a-eye/Output/metadata/sub_metadata_all.csv'
csv_axial_length_r = '/mnt/sda1/Repos/a-eye/Output/axial_length/nnunet/axial_length_nnunet_3D_N1210.csv'
csv_axial_length_l = '/mnt/sda1/Repos/a-eye/Output/axial_length/nnunet/axial_length_nnunet_3D_N1210_left_eyes.csv'

# Pandas read csv
pd_metadata = pd.read_csv(csv_metadata)
pd_axial_length_r = pd.read_csv(csv_axial_length_r)
pd_axial_length_l = pd.read_csv(csv_axial_length_l)

# Merge both dataframes
pd_axial_length = pd.merge(
    pd_axial_length_r,
    pd_axial_length_l,
    on='Subject',
    how='outer',
    suffixes=('_r', '_l')
)

# Clean Subject column: remove "AEye_" prefix and ".nii.gz" suffix
pd_axial_length['Subject'] = pd_axial_length['Subject'].str.replace('AEye_', '').str.replace('.nii.gz', '')


/tmp/ipykernel_483176/3426245269.py:24: FutureWarning: The default value of regex will change from True to False in a future version.
  pd_axial_length['Subject'] = pd_axial_length['Subject'].str.replace('AEye_', '').str.replace('.nii.gz', '')
